# 0. Imports

## 0.1 Packages

In [182]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

In [183]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")

# 1. Taxonomy Uniformization

## 1.1. Code

### 1.1.1. GBIF: Names Check

#### 1.1.1.1. Uniformization

In [184]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_1(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return species
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_1(species_list))

#### 1.1.1.2. Filter

In [185]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_2(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_2(species_list))

### 1.1.2. Global Names Verifier: Cross-check

#### 1.1.2.1. Uniformization

In [187]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_1(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name if accepted_name != "" else species
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_1(species_list))

#### 1.1.2.2. Filter

In [186]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_2(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_2(species_list))

### 1.1.3. GBIF: Family Extract

In [188]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Family(session, species):
    genus = species.split()[0]
    url = f"https://api.gbif.org/v1/species/match?name={genus}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('family'):
                return data.get('family')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Family_Sessions(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Family(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Family_Extract(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Family_Sessions(species_list))

### 1.1.4. GBIF: Lepidoptera Cross-check

In [189]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Lepidoptera(session, family):
    
    url = f"https://api.gbif.org/v1/species/match?name={family}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('order') == 'Lepidoptera':
                return 1
            elif data.get('order') is None:
                return None
            else:
                return 0
            
    except Exception as e:
        print(f"Error fetching data for {family}: {e}")
        return False

async def GBIF_Lepidoptera_Sessions(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Lepidoptera(session, family) for family in family_list]
        return await asyncio.gather(*tasks)

def GBIF_Lepidoptera_Extract(family_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Lepidoptera_Sessions(family_list))

## 1.2. Data

In [199]:
ObsList['Species'] = ObsList['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [191]:
ObsList['VNF_GBIF_VNF'] = VNF_Extractor_2(GBIF_Extractor_1(VNF_Extractor_1(ObsList['Species'])))
ObsList['GBIF_VNF_GBIF'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(ObsList['Species'])))
ObsList['VNF_GBIF'] = GBIF_Extractor_2(VNF_Extractor_1(ObsList['Species']))
ObsList['GBIF_VNF'] = VNF_Extractor_2(GBIF_Extractor_1(ObsList['Species']))

Request failed for species 'Scirpophaga incertulas': Connection timeout to host https://verifier.globalnames.org/api/v1/verifications/Scirpophaga%20incertulas?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5
Request failed for species 'Pectinophora gossypiella': Connection timeout to host https://verifier.globalnames.org/api/v1/verifications/Pectinophora%20gossypiella?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5
Request failed for species 'Phthorimaea absoluta': Connection timeout to host https://verifier.globalnames.org/api/v1/verifications/Phthorimaea%20absoluta?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5


In [ ]:
ObsList['VNF_GBIF'] = ObsList['VNF_GBIF'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

ObsList['VNF_GBIF_VNF'] = ObsList['VNF_GBIF_VNF'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

ObsList['GBIF_VNF'] = ObsList['GBIF_VNF'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

ObsList['GBIF_VNF_GBIF'] = ObsList['GBIF_VNF_GBIF'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)


In [202]:
ObsList['VNF_GBIF'].value_counts()

VNF_GBIF
Spodoptera frugiperda       676
Helicoverpa armigera        545
Phthorimaea operculella     509
Phthorimaea absoluta        448
Phyllocnistis citrella      415
                           ... 
Microsphecia tineiformis      1
Acraea macarista              1
Anapisa holobrunnea           1
Apisa metarctiodes            1
Archinadata aurivilliusi      1
Name: count, Length: 1373, dtype: int64

In [203]:
ObsList['VNF_GBIF_VNF'].value_counts()

VNF_GBIF_VNF
Spodoptera frugiperda      676
                           616
Helicoverpa armigera       545
Phthorimaea operculella    509
Tuta absoluta              448
                          ... 
Libythea celtis              1
Chaliopsis junodi            1
Junonia atlites              1
Hipparchia statilinus        1
Symbrenthia hypselis         1
Name: count, Length: 1310, dtype: int64

In [204]:
ObsList['GBIF_VNF'].value_counts()

GBIF_VNF
Spodoptera frugiperda        676
                             616
Helicoverpa armigera         545
Phthorimaea operculella      510
Tuta absoluta                448
                            ... 
Symbrenthia hypselis           1
Pareuchaetes aurata            1
Uripao albizonata              1
Stenoglene plagiatus           1
Rhypopteryx rubripunctata      1
Name: count, Length: 1309, dtype: int64

In [205]:
ObsList['GBIF_VNF_GBIF'].value_counts()

GBIF_VNF_GBIF
Spodoptera frugiperda      676
Helicoverpa armigera       545
Phthorimaea operculella    510
Phthorimaea absoluta       447
Phyllocnistis citrella     415
                          ... 
Doleschallia bisaltide       1
Parornix pfaffenzelleri      1
Coptodisca juglandiella      1
Daphaenisca inexpectata      1
Phyllonorycter viciae        1
Name: count, Length: 1374, dtype: int64

# 2. Taxonomy Extract

In [187]:
Taxonomy = pd.DataFrame(ObsList['Species'].unique(), columns=['Species'])

In [188]:
Taxonomy['Species'] = Taxonomy['Species'].apply(lambda x: ' '.join(x.split()[:2]))

## 2.1 Accepted Species

In [189]:
Taxonomy['AcceptedSpecies'] = check_species(Taxonomy['Species'])

In [190]:
Taxonomy[Taxonomy['AcceptedSpecies'].isna()]

,Species,AcceptedSpecies
326,Heliocheilus cystiphora,None
355,Rheumaptera affirmata,None
362,Spodoptera sunia,None
363,Heliocontia margana,None
402,Trissodoris guamensis,None
585,Semiothisa santaremaria,None
628,Limenitis camilla,None
645,Phalaenophana fadusalis,None
661,Eumeta japonica,None
821,Agonopterix umbellana,None


In [191]:
Taxonomy['AcceptedSpecies'] = Taxonomy['AcceptedSpecies'].fillna(Taxonomy['Species'])
Taxonomy['AcceptedSpecies'] = Taxonomy['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]))

Taxonomy.reset_index(drop=True, inplace=True)

## 2.2 Family

In [192]:
Taxonomy['Family'] = check_family(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

## 2.3 Genus

In [193]:
Taxonomy['Genus'] = check_genus(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

## 2.4 Lepidoptera Cross-Check

In [194]:
Taxonomy['Lepidoptera'] = check_lepidoptera(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

In [195]:
Taxonomy[Taxonomy['Lepidoptera'] == 0]

,Species,AcceptedSpecies,Family,Genus,Lepidoptera


In [196]:
Taxonomy.drop(columns=['Lepidoptera'], inplace=True)

In [197]:
Taxonomy.drop_duplicates(inplace=True)

## 2.5 Export

In [198]:
Taxonomy.to_csv(r'../Data Raw/TaxonomyRaw.csv', index=False)

# 3. Natives Taxonomy

In [199]:
Natives = pd.read_csv(r'../Data Raw/NativeRaw.csv', sep=';')

In [200]:
Natives.drop('AcceptedSpecies', axis=1, inplace=True)

In [204]:
Natives['AcceptedSpecies'] = check_species(Natives['Species'])
Natives['AcceptedSpecies'] = Natives['AcceptedSpecies'].fillna(Natives['Species'])


In [206]:
Natives.to_csv(r'../Data Raw/NativeRaw.csv', sep=';')